# Music Generation I

## Exercise 1 [Markov Models, 4 points]

In [70]:

import os
import pathlib
import random
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Protocol, Tuple, Dict

import math
import music21
import numpy as np
import torch
import torch.nn.utils as F
from music21 import converter
from music21.stream import Score
from pomegranate.markov_chain import MarkovChain

In [71]:
current_dir = pathlib.Path.cwd()
data_path = current_dir.joinpath("data")
output_path = data_path.joinpath("output")

# File extensions
ABC_FILE_EXTENSION = ".abc"
MUSIC_XML_FILE_EXTENSION = ".xml"
MIDI_FILE_EXTENSION = ".midi"

# Data
STYLES = ["french", "irish_folk"]
ORDERS = [1, 2, 4]

# Constants
N_BARS = 16

In [72]:

@dataclass(frozen=True)
class State:
    note: int
    duration: int

    #score_position: Optional[int] = None
    #bar_position: Optional[int] = None
    #articulation: Optional[int] = None
    # add fields as you discover them

    def to_music21(self):
        if self.note == 0:
            return music21.note.Rest(quarterLength=self.duration)
        else:
            return music21.note.Note(self.note, quarterLength=self.duration)


def state_from_music21(note: music21.note.GeneralNote) -> State:
    if isinstance(note, music21.note.Note):
        return State(note=note.pitch.midi, duration=note.duration.quarterLength)
    elif isinstance(note, music21.note.Rest):
        return State(note=0, duration=note.duration.quarterLength)
    else:
        raise NotImplementedError(f"state from music21 {note}")


class Embedding(Protocol):
    def initialize_embedding(self, states: list[State]) -> None:
        """ Initializes embedding. """

    def encode(self, state: State) -> int:
        """Project a State to a discrete token."""

    def decode(self, idx: int) -> State:
        """Optional: reconstruct a State (partial is acceptable)."""

    def get_eos_idx(self) -> int:
        """Index used for end of sentence and padding."""

    def __len__(self) -> int:
        """Vocabulary size."""


class DistinctStateEmbedding(Embedding):
    """
    One embedding for each distinct symbol.
    """

    def __init__(self):
        self.state2idx = None
        self.idx2state = None
        self.eos_idx = 0

    def initialize_embedding(self, states: list[State]):
        unique_states = list(set(states))
        self.state2idx = {s: i + 1 for i, s in enumerate(unique_states)}
        self.idx2state = {i + 1: s for i, s in enumerate(unique_states)}

    def encode(self, state: State) -> int:
        return self.state2idx[state]

    def decode(self, idx: int) -> Optional[State]:
        if idx == self.eos_idx:
            return None
        assert idx <= len(self.idx2state), f"idx out of range (idx {idx} > {len(self.idx2state)} vocab_size)"
        return self.idx2state[idx]

    def get_eos_idx(self) -> int:
        return self.eos_idx

    def __len__(self) -> int:
        return len(self.state2idx)

In [73]:
def get_state_seq(file: pathlib.Path) -> list[State]:
    """
    For now only storing pitch and duration of the notes and rests
    :param file:
    :return:
    """
    states = []

    score = converter.parse(file)
    for el in score.recurse():
        if isinstance(el, music21.note.Note) or isinstance(el, music21.note.Rest):
            states.append(state_from_music21(el))
        else:
            continue

    return states


def get_samples(style_dir: pathlib.Path, embedding_class: type[Embedding]) -> Tuple[torch.Tensor, Embedding]:
    # Get state sequences
    state_sequences: list[list[State]] = []
    all_states: list[State] = []
    for file in sorted(style_dir.glob(f"*{ABC_FILE_EXTENSION}")):
        state_seq = get_state_seq(file)
        state_sequences.append(state_seq)
        all_states.extend(state_seq)

    # Initialize embedding
    embedding = embedding_class()
    embedding.initialize_embedding(all_states)

    # Get tensor
    tensor_list = [torch.tensor([embedding.encode(el) for el in seq]) for seq in state_sequences]
    X = F.rnn.pad_sequence(tensor_list, batch_first=True, padding_value=embedding.get_eos_idx())
    X = X.unsqueeze(-1)
    return X, embedding


def create_and_fit_markov_model(data: torch.Tensor, order) -> MarkovChain:
    model = MarkovChain(k=order)
    model.fit(data)
    return model


def sample_from_probs(probs: torch.Tensor, generator: torch.Generator, eos_idx: int) -> int:
    """
    probs: 1D tensor summing to 1
    """
    probs_wo_eos = probs.clone()
    probs_wo_eos[eos_idx] = 0.0
    total_mass = probs_wo_eos.sum()
    if total_mass == 0:
        return eos_idx
    else:
        probs_wo_eos /= total_mass
        return torch.multinomial(probs_wo_eos, num_samples=1, generator=generator).item()


def get_random_initial_state(n_samples: int, vocab_size: int, seed: int) -> list[int]:
    """
    :param n_samples:
    :param vocab_size:
    :return:
    """
    random.seed(seed)
    return [random.randint(0, vocab_size) for _ in range(n_samples)]


def get_duration_of_idx_seq(seq: list[int], embedding: Embedding) -> float:
    return sum([embedding.decode(i).duration for i in seq])


def generate_seq(model: MarkovChain, embedding: Embedding, seed: int, n_bars: int = 4,
                 max_samples: Optional[int] = np.inf, ) -> \
        list[int]:
    seq = get_random_initial_state(n_samples=model.k, vocab_size=len(embedding), seed=seed)

    if seed is not None:
        gen = torch.Generator()
        gen.manual_seed(seed)
    else:
        gen = None

    target_score_duration = n_bars * 4  #ASSUMING 4/4 and quarternote duration!!    

    i = 0
    score_duration = get_duration_of_idx_seq(seq, embedding)
    while score_duration < target_score_duration:
        dist = model.distributions[model.k].probs[0]
        context = seq[-model.k:]
        probs_i = dist[tuple(context)]
        i_idx = sample_from_probs(probs_i, generator=gen, eos_idx=embedding.get_eos_idx())
        seq.append(i_idx)
        i += 1
        new_state = embedding.decode(i_idx)
        if new_state is None:
            print(f"Unable to generate a longer piece, final size={score_duration}")
            break
        else:
            score_duration += embedding.decode(i_idx).duration
        if max_samples is not None and i >= max_samples:
            break

    return seq


# def trim_part_to_n_measures(part: music21.stream.Part, n: int):
#     """
#     Keeps only the first n measures of a music21 Part.
#     Modifies the Part in place.
#     """
#     measures = list(part.measures(0, None))
#
#     if len(measures) <= n:
#         print(f"{len(measures)}, {n}")
#         return  # nothing to trim
#
#     for m in measures[n:]:
#         part.remove(m)
#
#     print(len(part.measures(0, None)))


def export_results(idx_seq: List[int], embedding: Embedding, name: str,
                   n_bars: int,
                   out_path: Optional[pathlib.Path] = None,
                   store_xml: bool = False,
                   store_midi: bool = False,
                   store_score_image: bool = False,
                   store_score_audio: bool = False,
                   ) -> music21.stream.Score:
    state_seq = [s for s in [embedding.decode(x) for x in idx_seq] if s is not None]

    score = state_list_to_score(state_seq)

    # Metadata
    score.insert(0, music21.metadata.Metadata())
    score.metadata.title = name
    score.metadata.composer = "generated using a Markov Chain"

    if out_path is not None:
        # XML
        if store_xml:
            export_xml(name, out_path, score)

        # MIDI
        # if store_midi:
        #     score_tmp_path = pathlib.Path(score.write('midi'))
        #     midi_str = score_tmp_path.read_text()
        #     file_path = out_path.joinpath(f"{name}{MIDI_FILE_EXTENSION}")
        #     with open(file_path, "w+") as f:
        #         f.write(midi_str)

        # SCORE
        if store_score_image:
            pass  # TODO: !!!

        # AUDIO
        if store_score_audio:
            pass  # TODO: !!!

    return score


def export_xml(name: str, out_path: Path, score: Score):
    score_tmp_path = pathlib.Path(score.write('musicxml'))
    music_xml_str = score_tmp_path.read_text()
    file_path = out_path.joinpath(f"{name}{MUSIC_XML_FILE_EXTENSION}")
    with open(file_path, "w+") as f:
        f.write(music_xml_str)


def state_list_to_score(state_seq: list[State]) -> music21.stream.Score:
    stream = music21.stream.Stream()
    for state in state_seq:
        stream.append(state.to_music21())

    score = music21.stream.Score()
    part = music21.stream.Part()
    part.append(stream.makeMeasures())
    # trim_part_to_n_measures(part, n_bars)
    score.append(part)
    return score


In [74]:
def get_seqs(styles: List[str],
             orders: List[int],
             embedding_class: Optional[type[Embedding]] = DistinctStateEmbedding,
             verbose: bool = False,
             max_samples: int = 250,
             n_bars: int = 16,
             seed: Optional[int] = 42,
             save_files: bool = True) -> Tuple[
    pathlib.Path, Dict[str, MarkovChain], Dict[str, type[Embedding]], Dict[str, music21.stream.Score]]:
    # OUT PATH
    if save_files:
        out_folder_name = f"{datetime.now().strftime('%y%m%d-%H%M%S')}"
        out_folder_path = output_path.joinpath(out_folder_name)
        if not os.path.exists(out_folder_path):
            os.mkdir(out_folder_path)
    else:
        out_folder_path = None

    models = {}
    scores = {}
    embeddings = {}
    for style in styles:
        # DATA
        if verbose:
            print(f"\nProcessing -> {style}")
        data, embedding = get_samples(data_path.joinpath(style), embedding_class)
        embeddings[style] = embedding
        if verbose:
            print(f"\tVocabulary size: {len(embedding)}")
            #print(f"\tdata shape: {data.shape}")

        for order in orders:
            model_name = get_model_name(style, order)

            # TRAINING
            if verbose:
                print(f"\n\tTraining model with order {order}", end="")
            model = create_and_fit_markov_model(data=data, order=order)
            models[model_name] = model
            if verbose:
                print("\t -> Done!")

            # INFERENCE
            idx_seq = generate_seq(model, max_samples=max_samples, n_bars=n_bars, embedding=embedding, seed=seed)
            #if verbose:
            #    print(f"\t{idx_seq}")

            # EXPORT
            score = export_results(idx_seq, embedding=embedding, n_bars=n_bars, out_path=out_folder_path,
                                   name=model_name, store_xml=True)
            scores[model_name] = score

    return out_folder_path, models, embeddings, scores


def get_model_name(style: str, order: int) -> str:
    return f"{style}-{order}"


In [75]:
out_folder_path, models, embeddings, scores = get_seqs(STYLES, ORDERS, n_bars=16, verbose=True)

	 -> Done!


## Exercise 2 [Evaluation, 2 points]

In [76]:
class EvaluationMethod(Protocol):

    @staticmethod
    def evaluate(score: music21.stream.Score) -> float:
        """Returns a scalar given a piece."""

In [77]:
class MelodicSmoothness(EvaluationMethod):
    """
    Penalizes large intervals.
    Calculated averaging average pitch distance between intervals and exp.
    To maximize.
    """

    @staticmethod
    def evaluate(score: music21.stream.Score, sigma: float = 3.5, var: float = 3) -> float:
        """
        :param score:
        :param tau: with 3.0 you add tolerance to up to avg distance of 6. More than that seems too much
        :return:
        """
        notes = list(score.recurse().notes)
        prev_pitch = notes[0].pitch.midi
        res = 0
        for n in notes:
            if isinstance(n, music21.note.Note):
                res += abs(prev_pitch - n.pitch.midi)
                prev_pitch = n.pitch.midi

        abs_mean = res / (len(notes) - 1)
        return math.exp(-(abs_mean - sigma) ** 2 / (2 * var ** 2))


print(f"Using evaluation method {EvaluationMethod.__name__}")
for score_name, score in scores.items():
    score = MelodicSmoothness.evaluate(score)
    print(f"\t{score_name} -> {round(score, 5)}")

Using evaluation method EvaluationMethod
	french-1 -> 0.89707
	french-2 -> 0.9956
	french-4 -> 0.98621
	irish_folk-1 -> 0.99602
	irish_folk-2 -> 0.97947
	irish_folk-4 -> 0.47015


In [78]:
#TODO: add some rythm metric

In [79]:
#TODO: add some harmoic metric

In [80]:
#TODO: add some piece duration metric

In [81]:
#TODO: combine them into a metric of goodness (normalize all between 0 and 1 or so...

## Exercise 3 [Evolutionary Algorithms, 4 points]

In [82]:
# Mutations
class Mutation(Protocol):
    PITCH_RANGE = list(range(48, 88)) + [0]  # MIDI pitches: C3–E6, plus 0 for rest
    DURATION_RANGE = [0.25 * i for i in range(1, 9)]  # [0.25, 0.5, 1.0, 1.5, 2.0]

    COMPOSER_EA_STRING = "generated using a Markov Chain\nand modified with an Evolutionary Algorithm"

    @staticmethod
    def mutate(score: music21.stream.Score) -> music21.stream.Score:
        """Generate mutation from an original individual."""

    @staticmethod
    def get_op() -> str:
        """Get a op str."""


def get_state_list(score: music21.stream.Score) -> List[State]:
    assert len(score.getElementsByClass(music21.stream.Part))

    state_list = []
    elements = score.parts[0].flat.notesAndRests.stream()
    for el in elements:
        if isinstance(el, music21.note.Note) or isinstance(el, music21.note.Rest):
            state_list.append(state_from_music21(el))
    return state_list


class AddRandomNote(Mutation):

    @staticmethod
    def mutate(score: music21.stream.Score) -> music21.stream.Score:
        state_list = get_state_list(score)

        pitch = random.choice(Mutation.PITCH_RANGE)
        duration = random.choice(Mutation.DURATION_RANGE)
        position = random.randrange(0, len(state_list))

        new_state = State(note=pitch, duration=duration)
        state_list.insert(position, new_state)

        new_score = state_list_to_score(state_list)

        # Metadata
        new_score.insert(0, music21.metadata.Metadata())
        new_score.metadata.title = f"{score.metadata.title},{AddRandomNote.get_op()}"
        new_score.metadata.composer = Mutation.COMPOSER_EA_STRING

        return new_score

    @staticmethod
    def get_op() -> str:
        return "N"


class RemoveRandomNote(Mutation):
    @staticmethod
    def mutate(score: music21.stream.Score) -> music21.stream.Score:
        state_list = get_state_list(score)

        idx = random.randrange(0, len(state_list))
        state_list.pop(idx)

        new_score = state_list_to_score(state_list)

        # Metadata
        new_score.insert(0, music21.metadata.Metadata())
        new_score.metadata.title = f"{score.metadata.title},{RemoveRandomNote.get_op()}"
        new_score.metadata.composer = Mutation.COMPOSER_EA_STRING

        return new_score

    @staticmethod
    def get_op() -> str:
        return "D"


class ChangePitchOfRandomNote(Mutation):
    @staticmethod
    def mutate(score: music21.stream.Score) -> music21.stream.Score:
        state_list = get_state_list(score)

        pitch = random.choice(Mutation.PITCH_RANGE)
        duration = random.choice(Mutation.DURATION_RANGE)
        position = random.randrange(0, len(state_list))

        old_state = state_list[position]

        opt = random.randint(0, 2)
        if opt == 0:
            new_state = State(note=pitch, duration=duration)
        elif opt == 1:
            new_state = State(note=pitch, duration=old_state.duration)
        elif opt == 2:
            new_state = State(note=old_state.note, duration=duration)
        else:
            raise ValueError(f"Invalid op: {opt}")

        state_list[position] = new_state

        new_score = state_list_to_score(state_list)

        # Metadata
        new_score.insert(0, music21.metadata.Metadata())
        new_score.metadata.title = f"{score.metadata.title},{ChangePitchOfRandomNote.get_op()}"
        new_score.metadata.composer = Mutation.COMPOSER_EA_STRING

        return new_score

    @staticmethod
    def get_op() -> str:
        return "S"


In [83]:
N_GENERATIONS = 100
N_INDIVIDUALS_TO_KEEP = 10
N_CHILD_PER_OG_INDIVIDUAL = 1

MUTATIONS = [AddRandomNote, RemoveRandomNote, ChangePitchOfRandomNote]

In [84]:
# Population

def generate_score_from_model(model: MarkovChain, embedding: Embedding, seed: int = 42,
                              name: str = "original_individual") -> music21.stream.Score:
    idx_seq = generate_seq(model, n_bars=N_BARS, embedding=embedding, seed=seed)
    score = export_results(idx_seq, embedding=embedding, n_bars=N_BARS, name=name)
    return score


def get_population(models: list[MarkovChain], embeddings: List[Embedding], n: int = N_INDIVIDUALS_TO_KEEP,
                   seed=42) -> list[music21.stream.Score]:
    i_per_model = int(n / len(models))
    assert i_per_model * len(models) == n, "n is not divisible by the number of models"
    print(f"i_per_model: {i_per_model}")

    population = []
    i_idx = 0
    for m_idx, model in enumerate(models):
        for i in range(i_per_model):
            specific_seed = seed + i_idx
            population.append(generate_score_from_model(model, embeddings[m_idx], seed=specific_seed, name=f"p{i_idx}"))
            i_idx += 1

    return population

In [85]:
# Evolution Loop

def apply_random_mutation(individual: music21.stream.Score, mutations: list[type[Mutation]]) -> music21.stream.Score:
    mutation = random.choice(mutations)
    return mutation.mutate(individual)


def top_n(candidates, scores, n) -> list[music21.stream.Score]:
    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True  # higher score = better
    )
    top = ranked[:n]
    top_candidates, top_scores = zip(*top)
    return list(top_candidates)  #, list(top_scores)


def print_population(scores: List[music21.stream.Score]):
    print(" -> ", end="")
    for score in scores:
        print(score.metadata.title, ", ", end="")
    print(" <- ")


def ea_loop(
        population: list[music21.stream.Score],
        fitness_function: type[EvaluationMethod],

        mutations=None,
        n_generations: int = N_GENERATIONS,
        n_individuals_to_keep: int = N_INDIVIDUALS_TO_KEEP,
        n_child_per_og_individual: int = N_CHILD_PER_OG_INDIVIDUAL,
        seed: int = 42,
        verbose: bool = False
) -> Tuple[list[music21.stream.Score], list[float], list[float]]:
    random.seed(seed)
    if mutations is None:
        mutations = MUTATIONS
    current_population = population.copy()

    print_population(current_population)
    initial_scores = [fitness_function.evaluate(i) for i in population]

    for generation in range(n_generations):
        if verbose:
            print(f"Generation {generation}...")

        # Generate candidates
        if verbose:
            print("Generating candidates...", end="")
        candidates = current_population.copy()
        for idx, individual in enumerate(current_population):
            for _ in range(n_child_per_og_individual):
                new_child = apply_random_mutation(individual, mutations)
                candidates.append(new_child)
            if verbose:
                print(f" {idx}, ", end="")
        if verbose:
            print()

        # Candidate selection
        if verbose:
            print("Selecting candidates...", end="")
        scores = [fitness_function.evaluate(c) for c in candidates]
        selected_candidates = top_n(candidates, scores, n=n_individuals_to_keep)

        current_population = selected_candidates.copy()
        print_population(current_population)

    final_scores = [fitness_function.evaluate(i) for i in current_population]

    return current_population, initial_scores, final_scores

In [86]:
# FRENCH!

"""
french-1 -> 0.81857
    french-2 -> 0.96663
french-4 -> 1.0
"""
models_for_ea1 = [models[get_model_name(STYLES[0], 2)]]
embeddings_for_ea1 = [embeddings[STYLES[0]]]
population1 = get_population(models_for_ea1, embeddings_for_ea1, n=N_INDIVIDUALS_TO_KEEP, seed=42)
fitness_function1 = MelodicSmoothness

assert len(
    population1) == N_INDIVIDUALS_TO_KEEP, f"Starting with population of size {len(population1)}!! should be {N_INDIVIDUALS_TO_KEEP}!"
final_population, ini_scores, fini_scores = ea_loop(
    population=population1,
    fitness_function=fitness_function1,
    seed=42,
    verbose=True
)

ea_output = out_folder_path.joinpath("ea_french")
os.makedirs(ea_output, exist_ok=True)
for idx, score in enumerate(final_population):
    export_xml(name=f"i_{idx}", out_path=ea_output, score=score)

 7,  8,  9, 
Selecting candidates... -> p0,N,D,D,N,N , p0,N,D,S,N , p0,N,D,S,N,S , p0,N,D,D,N,N,S , p0,N,D,S,N,S,S , p0,N,D,N , p0,N,D,S,N,S,S,S , p0,N,D,S,N,S , p0,N,D,D,N,N,S,S , p0,N,D,S,N,S,S ,  <- 
Generation 98...
Generating candidates... 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 
Selecting candidates... -> p0,N,D,D,N,N , p0,N,D,S,N , p0,N,D,S,N,S , p0,N,D,D,N,N,S , p0,N,D,S,N,S,S , p0,N,D,N , p0,N,D,S,N,S,S,S , p0,N,D,S,N,S , p0,N,D,D,N,N,S,S , p0,N,D,S,N,S,S ,  <- 
Generation 99...
Generating candidates... 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 
Selecting candidates... -> p0,N,D,D,N,N , p0,N,D,S,N , p0,N,D,S,N,S , p0,N,D,D,N,N,S , p0,N,D,S,N,S,S , p0,N,D,N , p0,N,D,S,N,S,S,S , p0,N,D,S,N,S , p0,N,D,D,N,N,S,S , p0,N,D,S,N,S,S ,  <- 


In [87]:
# IRISH FOLK
"""
    irish_folk-1 -> 0.96777
irish_folk-2 -> 0.93371
irish_folk-4 -> 0.56903
"""
models_for_ea2 = [models[get_model_name(STYLES[1], 1)]]
embeddings_for_ea2 = [embeddings[STYLES[1]]]
population2 = get_population(models_for_ea2, embeddings_for_ea2, n=N_INDIVIDUALS_TO_KEEP, seed=42)
fitness_function2 = MelodicSmoothness

assert len(
    population2) == N_INDIVIDUALS_TO_KEEP, f"Starting with population of size {len(population2)}!! should be {N_INDIVIDUALS_TO_KEEP}!"
final_population, ini_scores, fini_scores = ea_loop(
    population=population2,
    fitness_function=fitness_function2,
    seed=42,
    verbose=True
)

ea_output = out_folder_path.joinpath("ea_irish_folk")
os.makedirs(ea_output, exist_ok=True)
for idx, score in enumerate(final_population):
    export_xml(name=f"i_{idx}", out_path=ea_output, score=score)

 1,  2,  3,  4,  5,  6,  7,  8,  9, 
Selecting candidates... -> p2,N,D,N,N,D , p0,D,S,S,D,D,D , p9,N,N,S,N , p0,D,S,S,D,D,D,S , p2,N,D,N,N,D,S , p0,D,S,S,D,D,D,S , p0,D,S,S,D,D,D , p2,N,D,N,N,D,S , p9,N,N,S,N,S , p2,N,D,N,N,D,S ,  <- 
Generation 99...
Generating candidates... 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 
Selecting candidates... -> p2,N,D,N,N,D , p0,D,S,S,D,D,D , p9,N,N,S,N , p0,D,S,S,D,D,D,S , p2,N,D,N,N,D,S , p0,D,S,S,D,D,D,S , p0,D,S,S,D,D,D , p2,N,D,N,N,D,S , p9,N,N,S,N,S , p2,N,D,N,N,D,S ,  <- 
